<!-- NB_DOC_INTRO_V1 -->
# Metriques d'evaluation — Reference exhaustive

> 📚 **Doc thematique** : [docs/03_ML.md](docs/03_ML.md) (Machine Learning classique)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

Cette cheat-sheet **rassemble en un seul endroit toutes les metriques d'evaluation** utiles en data science, organisees par famille de probleme. L'objectif n'est pas seulement de lister `accuracy`, `F1`, `MSE` : c'est de couvrir **les ~80 metriques** qu'on croise reellement en production (calibration, ranking IR, survival, forecasting, NLP, vision, fairness).

Pour **chaque metrique** : definition mathematique, formule, code sklearn (ou autre lib), interpretation, plage de valeurs, et **piege courant** d'usage. Les choix de metriques sont **critiques** : une bonne metrique alignee sur le besoin metier vaut un meilleur modele. Inversement, optimiser une mauvaise metrique = degrader silencieusement la production.

**Quand consulter ce notebook ?**
- Avant de definir l'objectif d'optimisation d'un nouveau projet
- Quand on hesite entre 2 metriques pour reporter
- Pour comprendre une metrique vue dans un papier
- Pour configurer une `GridSearchCV(scoring=...)` ou `Trainer(metric_for_best_model=...)`

## Plan

1. Generation du jeu de donnees jouet
2. **Classification binaire** — accuracy, precision, recall, F1, F-beta, ROC-AUC, PR-AUC, MCC, Cohen's kappa, balanced accuracy, log-loss, Brier, KS, Gini, lift
3. **Classification multi-classe** — confusion matrix, top-k accuracy, micro/macro/weighted, Cohen's kappa quadratique
4. **Multi-label** — Hamming loss, Jaccard, subset accuracy, label ranking
5. **Regression** — MSE, RMSE, MAE, MAPE, sMAPE, MASE, R^2, adjusted R^2, explained variance, max error, MSLE, Huber, quantile loss, Tweedie
6. **Ranking / Information Retrieval** — MRR, MAP@k, NDCG@k, Precision@k, Recall@k, HitRate@k
7. **Calibration probabiliste** — reliability diagram, ECE, MCE, isotonic / Platt calibration
8. **Clustering** — silhouette, Davies-Bouldin, Calinski-Harabasz, ARI, AMI, NMI, V-measure, homogeneity, completeness
9. **Survival** — C-index, Brier score, IBS, time-dependent AUC
10. **Time series forecasting** — MASE, sMAPE, RMSSE, WAPE, OWA, pinball loss
11. **NLP generation** — BLEU, ROUGE, METEOR, chrF, BERTScore, perplexity
12. **Computer Vision** — IoU, mAP, pixel accuracy, Dice, PSNR, SSIM
13. **LLM / RAG** — faithfulness, answer relevancy, context precision / recall (ragas), LLM-as-judge (G-Eval)
14. **Fairness** — disparate impact, demographic parity, equalized odds, equal opportunity
15. **Tableau de selection** — quelle metrique pour quel probleme ?
16. **Pieges generaux** + references

> ⚠️ **Note importante** : Toutes les metriques de ce notebook sont **executables** avec leurs imports respectifs. Certains imports (NLP, vision, survival) necessitent des libs additionnelles — installes a la demande dans chaque section.


## 1. Installation et donnees jouet

Imports de base et generation de jeux de donnees synthetiques reproductibles pour illustrer chaque famille.


In [ ]:
# pip install -q scikit-learn numpy pandas matplotlib seaborn scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sklearn.datasets import make_classification, make_regression, make_blobs, make_multilabel_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LinearRegression

SEED = 42
np.random.seed(SEED)

# Affichage
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

print("Imports OK")

### Datasets jouets utilises dans ce notebook

- **Classif binaire** : `make_classification` desequilibre 80/20
- **Classif multi-classe** : `make_classification` 4 classes
- **Multi-label** : `make_multilabel_classification`
- **Regression** : `make_regression` avec bruit
- **Clustering** : `make_blobs`


In [ ]:
# === Classification binaire (desequilibree pour metriques) ===
X_bin, y_bin = make_classification(
    n_samples=2000, n_features=20, n_informative=10, n_redundant=5,
    weights=[0.8, 0.2],  # 80% classe 0, 20% classe 1
    flip_y=0.05, random_state=SEED,
)
X_bin_tr, X_bin_te, y_bin_tr, y_bin_te = train_test_split(X_bin, y_bin, test_size=0.3, stratify=y_bin, random_state=SEED)

clf_bin = LogisticRegression(max_iter=1000).fit(X_bin_tr, y_bin_tr)
y_bin_pred = clf_bin.predict(X_bin_te)
y_bin_proba = clf_bin.predict_proba(X_bin_te)[:, 1]

# === Classification multi-classe (4 classes equilibrees) ===
X_mc, y_mc = make_classification(
    n_samples=2000, n_features=20, n_informative=15, n_redundant=2,
    n_classes=4, n_clusters_per_class=1, random_state=SEED,
)
X_mc_tr, X_mc_te, y_mc_tr, y_mc_te = train_test_split(X_mc, y_mc, test_size=0.3, random_state=SEED)
clf_mc = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_mc_tr, y_mc_tr)
y_mc_pred = clf_mc.predict(X_mc_te)
y_mc_proba = clf_mc.predict_proba(X_mc_te)

# === Regression ===
X_reg, y_reg = make_regression(n_samples=2000, n_features=15, n_informative=10, noise=20, random_state=SEED)
X_reg_tr, X_reg_te, y_reg_tr, y_reg_te = train_test_split(X_reg, y_reg, test_size=0.3, random_state=SEED)
reg = RandomForestRegressor(n_estimators=100, random_state=SEED).fit(X_reg_tr, y_reg_tr)
y_reg_pred = reg.predict(X_reg_te)

print(f"Bin classif : train={len(y_bin_tr)}, test={len(y_bin_te)}, pos rate test={y_bin_te.mean():.2%}")
print(f"Multi classif: train={len(y_mc_tr)}, test={len(y_mc_te)}, classes={np.unique(y_mc_te).tolist()}")
print(f"Regression  : train={len(y_reg_tr)}, test={len(y_reg_te)}, y range=[{y_reg_te.min():.1f}, {y_reg_te.max():.1f}]")

---

## 2. Classification binaire

La famille la plus riche. Avant tout, **comprendre la matrice de confusion** :

```
                 Predit Positif    Predit Negatif
Reel Positif     TP                FN              => Sensibilite = TP / (TP+FN)
Reel Negatif     FP                TN              => Specificite = TN / (TN+FP)
```

- **TP** (True Positive) : reel = 1, predit = 1
- **TN** (True Negative) : reel = 0, predit = 0
- **FP** (False Positive) : reel = 0, predit = 1 = "fausse alerte"
- **FN** (False Negative) : reel = 1, predit = 0 = "rate"

Toutes les metriques de classification binaire se calculent depuis ces 4 nombres ou depuis la fonction `predict_proba`.

### Tableau recapitulatif des metriques


| Metrique | Formule | Plage | Sensible au desequilibre ? | Threshold ? | Quand l'utiliser |
|---|---|---|---|---|---|
| **Accuracy** | `(TP+TN) / N` | [0, 1] | ❌ Tres | ✅ | Classes equilibrees uniquement |
| **Balanced accuracy** | `(TPR + TNR) / 2` | [0, 1] | ✅ Robuste | ✅ | Desequilibre, alternative simple a F1 |
| **Precision** | `TP / (TP+FP)` | [0, 1] | ✅ | ✅ | Cout FP eleve (mail spam : ne PAS rejeter du vrai courrier) |
| **Recall / Sensibilite / TPR** | `TP / (TP+FN)` | [0, 1] | ✅ | ✅ | Cout FN eleve (cancer : ne PAS rater un cas) |
| **Specificite / TNR** | `TN / (TN+FP)` | [0, 1] | ✅ | ✅ | Symetrique du recall |
| **F1** | `2*P*R/(P+R)` | [0, 1] | ✅ | ✅ | Compromis precision/recall, baseline |
| **F-beta** | `(1+β²)PR / (β²P+R)` | [0, 1] | ✅ | ✅ | Pondere precision/recall (β>1 favorise recall) |
| **ROC-AUC** | `int TPR d(FPR)` | [0, 1] | Moyennement | ❌ | Classement global, comparer 2 modeles |
| **PR-AUC** (Average Precision) | `int P d(R)` | [0, 1] | ✅ Robuste | ❌ | Desequilibre extreme, classes rares |
| **MCC** (Matthews Correlation) | `(TP·TN - FP·FN)/√...` | [-1, 1] | ✅ Robuste | ✅ | Resume balance tres compacte, recommande |
| **Cohen's kappa** | `(p_o - p_e)/(1 - p_e)` | [-1, 1] | ✅ | ✅ | Accord versus hasard |
| **Log loss** | `-Σ y log(p)` | [0, ∞[ | ✅ | ❌ | Eval de proba calibree |
| **Brier score** | `Σ (p - y)² / N` | [0, 1] | ✅ | ❌ | Eval proba + calibration |
| **KS statistic** | `max|F1(s)-F0(s)|` | [0, 1] | ✅ | ❌ | Risque credit, separabilite distrib |
| **Gini** | `2·AUC - 1` | [-1, 1] | Moyennement | ❌ | Equivalent ROC-AUC scale (-1, 1) |

> 🎯 **Regle de selection** :
> - Classes equilibrees + classement → **ROC-AUC**
> - Classes desequilibrees → **PR-AUC** + **MCC**
> - Decision avec cout asymetrique → **F-beta** ou **precision-recall a un threshold business**
> - Proba calibrees pour decision → **Brier score** + **calibration plot**


### 2.1 Metriques scalaires basees sur predict (threshold = 0.5 par defaut)


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, fbeta_score, matthews_corrcoef, cohen_kappa_score,
    confusion_matrix, classification_report,
)

# Matrice de confusion
cm = confusion_matrix(y_bin_te, y_bin_pred)
tn, fp, fn, tp = cm.ravel()
print(f"Confusion matrix:\n{cm}\nTN={tn} FP={fp} FN={fn} TP={tp}\n")

# Tableau de metriques
metrics = {
    "Accuracy":          accuracy_score(y_bin_te, y_bin_pred),
    "Balanced accuracy": balanced_accuracy_score(y_bin_te, y_bin_pred),
    "Precision":         precision_score(y_bin_te, y_bin_pred),
    "Recall (TPR)":      recall_score(y_bin_te, y_bin_pred),
    "Specificity (TNR)": tn / (tn + fp),
    "F1":                f1_score(y_bin_te, y_bin_pred),
    "F2 (β=2, recall++)": fbeta_score(y_bin_te, y_bin_pred, beta=2),
    "F0.5 (β=0.5, prec++)": fbeta_score(y_bin_te, y_bin_pred, beta=0.5),
    "MCC":               matthews_corrcoef(y_bin_te, y_bin_pred),
    "Cohen's kappa":     cohen_kappa_score(y_bin_te, y_bin_pred),
}
pd.DataFrame(metrics, index=["score"]).T

**Lecture** : sur ce dataset desequilibre (~20% positifs), une LogReg basique a une **accuracy ~85%** mais c'est trompeur — un dummy "toujours 0" donnerait 80%. Le **MCC** (souvent < 0.5 sur les cas reels difficiles) et le **F1** sont plus informatifs.

`classification_report` donne tout en un coup :


In [ ]:
print(classification_report(y_bin_te, y_bin_pred, target_names=["neg", "pos"], digits=3))

### 2.2 Metriques basees sur les probabilites (independantes du threshold)


In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    roc_curve, precision_recall_curve,
)

proba_metrics = {
    "ROC-AUC":              roc_auc_score(y_bin_te, y_bin_proba),
    "PR-AUC (Avg Precision)": average_precision_score(y_bin_te, y_bin_proba),
    "Log loss":             log_loss(y_bin_te, y_bin_proba),
    "Brier score":          brier_score_loss(y_bin_te, y_bin_proba),
    "Gini":                 2 * roc_auc_score(y_bin_te, y_bin_proba) - 1,
}
pd.DataFrame(proba_metrics, index=["score"]).T

### 2.3 KS statistic (risque credit)

**KS** (Kolmogorov-Smirnov) = distance maximale entre les CDF des positifs et negatifs. Standard en credit scoring : on cherche le score qui **separe** le plus les bons et les mauvais payeurs.


In [ ]:
from scipy.stats import ks_2samp

scores_pos = y_bin_proba[y_bin_te == 1]
scores_neg = y_bin_proba[y_bin_te == 0]
ks_stat, _ = ks_2samp(scores_pos, scores_neg)
print(f"KS statistic : {ks_stat:.4f}")
# Convention : KS > 0.3 = modele utilisable en credit
#              KS > 0.5 = modele fort

### 2.4 ROC curve et PR curve

- **ROC curve** : TPR (recall) vs FPR (1 - specificity), pour tous les threshold. Aire = AUC.
- **PR curve** : Precision vs Recall. Plus informative sur desequilibre.


In [ ]:
fpr, tpr, _ = roc_curve(y_bin_te, y_bin_proba)
prec, rec, _ = precision_recall_curve(y_bin_te, y_bin_proba)
auc_roc = roc_auc_score(y_bin_te, y_bin_proba)
ap = average_precision_score(y_bin_te, y_bin_proba)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(fpr, tpr, label=f"ROC (AUC={auc_roc:.3f})", lw=2)
ax1.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax1.set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC curve")
ax1.legend()

ax2.plot(rec, prec, label=f"PR (AP={ap:.3f})", lw=2, color="C1")
ax2.axhline(y_bin_te.mean(), color="k", ls="--", alpha=0.3, label=f"Baseline ({y_bin_te.mean():.2f})")
ax2.set(xlabel="Recall", ylabel="Precision", title="PR curve")
ax2.legend()
plt.tight_layout()
plt.show()

### 2.5 Lift et Gain (marketing / churn)

**Lift @ k%** = ratio entre la precision dans le top-k% scores et la precision globale.

> "En contactant les 10% mieux scores, mon taux de churn est X fois plus eleve que la moyenne."


In [ ]:
def lift_curve(y_true, y_score, n_bins=10):
    df = pd.DataFrame({"y": y_true, "s": y_score}).sort_values("s", ascending=False)
    df["decile"] = pd.qcut(df["s"].rank(method="first"), n_bins, labels=range(1, n_bins + 1))
    baseline = df["y"].mean()
    grouped = df.groupby("decile", observed=True)["y"].mean()
    lift = grouped / baseline
    return lift.sort_index(ascending=False)  # decile 10 = top 10%

lift = lift_curve(y_bin_te, y_bin_proba)
print("Lift par decile (10 = top 10%) :")
print(lift)
print(f"\nLift @ top-10% : {lift.iloc[0]:.2f}x  =>  le top 10% capture {lift.iloc[0]:.1f}x plus de positifs que le hasard")

### 2.6 Choix du threshold optimal

Le threshold 0.5 par defaut est rarement optimal en desequilibre. Methodes :

- **Youden's J statistic** : `argmax(TPR - FPR)` — maximise la difference avec le random
- **Coût metier** : `argmin(c_FN * FN + c_FP * FP)` — si on connait les coûts
- **Precision a recall minimum** : "je veux 90% de recall, quel threshold le plus precis ?"


In [ ]:
# Youden's J
fpr, tpr, thr_roc = roc_curve(y_bin_te, y_bin_proba)
j = tpr - fpr
best_thr_youden = thr_roc[np.argmax(j)]
print(f"Threshold optimal (Youden) : {best_thr_youden:.4f}")

# Cout metier (exemple : FN coute 10x plus que FP)
c_FN, c_FP = 10, 1
costs = []
for t in np.linspace(0, 1, 101):
    y_p = (y_bin_proba >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_bin_te, y_p).ravel()
    costs.append((t, c_FN * fn_ + c_FP * fp_))
costs = np.array(costs)
best_thr_cost = costs[costs[:, 1].argmin(), 0]
print(f"Threshold optimal (cout 10:1) : {best_thr_cost:.4f}")

---

## 3. Classification multi-classe (>= 3 classes)

Les metriques scalaires (precision, recall, F1) doivent etre **agregees** entre classes. 3 strategies :

| Strategie | Formule | Quand l'utiliser |
|---|---|---|
| **`micro`** | Agrege TP/FP/FN globalement avant calcul | = accuracy quand chaque sample = 1 label. Donne du poids aux classes frequentes |
| **`macro`** | Moyenne non-ponderee des metriques par classe | Traite toutes les classes egalement. Penalise un modele qui rate les rares |
| **`weighted`** | Moyenne ponderee par la frequence des classes | Comme micro mais pour multi-label aussi |
| **`samples`** | Moyenne par sample (multi-label seulement) | Multi-label |
| **`None`** | Renvoie par classe | Diagnostic |

> 🎯 **Choix** : si toutes les classes comptent egalement → `macro`. Si on suit l'effet global → `weighted` ou `micro`.

### 3.1 Metriques multi-classe


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    cohen_kappa_score, confusion_matrix, top_k_accuracy_score, classification_report,
)

results = []
for avg in ["micro", "macro", "weighted"]:
    results.append({
        "averaging": avg,
        "precision": precision_score(y_mc_te, y_mc_pred, average=avg),
        "recall":    recall_score(y_mc_te, y_mc_pred, average=avg),
        "f1":        f1_score(y_mc_te, y_mc_pred, average=avg),
    })
df_avg = pd.DataFrame(results).set_index("averaging")
print("Agreggation entre classes :")
print(df_avg, "\n")

# Par classe (None = pas d'agregation)
n_classes = len(np.unique(y_mc_te))
df_per_class = pd.DataFrame({
    "precision": precision_score(y_mc_te, y_mc_pred, average=None),
    "recall":    recall_score(y_mc_te, y_mc_pred, average=None),
    "f1":        f1_score(y_mc_te, y_mc_pred, average=None),
}, index=[f"class_{i}" for i in range(n_classes)])
print("Par classe :")
print(df_per_class, "\n")

print(f"Accuracy        : {accuracy_score(y_mc_te, y_mc_pred):.4f}")
print(f"Cohen kappa    : {cohen_kappa_score(y_mc_te, y_mc_pred):.4f}")
print(f"Top-2 accuracy : {top_k_accuracy_score(y_mc_te, y_mc_proba, k=2):.4f}")
print(f"Top-3 accuracy : {top_k_accuracy_score(y_mc_te, y_mc_proba, k=3):.4f}")

### 3.2 Quadratic Weighted Kappa (Cohen)

Variante du kappa qui **penalise davantage les erreurs eloignees** (ex : predire 1 etoile au lieu de 5 est pire que 4 au lieu de 5). Standard sur Kaggle pour ordinal classification.


In [ ]:
kappa_quad = cohen_kappa_score(y_mc_te, y_mc_pred, weights="quadratic")
kappa_linear = cohen_kappa_score(y_mc_te, y_mc_pred, weights="linear")
print(f"Cohen kappa (none)      : {cohen_kappa_score(y_mc_te, y_mc_pred):.4f}")
print(f"Cohen kappa (linear)    : {kappa_linear:.4f}")
print(f"Cohen kappa (quadratic) : {kappa_quad:.4f}  <- standard ordinal")

### 3.3 Multi-class ROC-AUC

3 strategies pour generaliser ROC-AUC :
- **OvR** (One-vs-Rest) : K AUCs, moyennes
- **OvO** (One-vs-One) : K(K-1)/2 AUCs (couples)
- Macro / weighted averaging


In [ ]:
from sklearn.metrics import roc_auc_score
print(f"ROC-AUC OvR macro    : {roc_auc_score(y_mc_te, y_mc_proba, multi_class='ovr', average='macro'):.4f}")
print(f"ROC-AUC OvR weighted : {roc_auc_score(y_mc_te, y_mc_proba, multi_class='ovr', average='weighted'):.4f}")
print(f"ROC-AUC OvO macro    : {roc_auc_score(y_mc_te, y_mc_proba, multi_class='ovo', average='macro'):.4f}")

### 3.4 Confusion matrix normalisee + viz

Plus parlant qu'un tableau brut.


In [ ]:
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
cm_raw = confusion_matrix(y_mc_te, y_mc_pred)
cm_norm = confusion_matrix(y_mc_te, y_mc_pred, normalize="true")

sns.heatmap(cm_raw, annot=True, fmt="d", cmap="Blues", ax=ax1)
ax1.set(title="Confusion matrix (counts)", xlabel="Predit", ylabel="Reel")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", ax=ax2, vmin=0, vmax=1)
ax2.set(title="Confusion matrix (normalisee par ligne)", xlabel="Predit", ylabel="Reel")
plt.tight_layout()
plt.show()

---

## 4. Classification multi-label

**Multi-label** ≠ multi-class. Ici chaque sample peut avoir **plusieurs labels** simultanement (ex: un article = ["tech", "ML", "tutorial"]).

| Metrique | Formule | Severite |
|---|---|---|
| **Subset accuracy** (exact match) | Fraction de samples avec **tous** les labels exacts | Tres severe |
| **Hamming loss** | Fraction de labels mal predits (par label, somme/N·K) | Lache (penalise par erreur, pas par sample) |
| **Jaccard score** | `|y∩y_pred| / |y∪y_pred|` (par sample) | Intermediaire |
| **F1 (samples)** | F1 calcule par sample puis moyenne | Intermediaire |
| **F1 (micro / macro)** | Comme multi-class | Selon strategie |


In [ ]:
from sklearn.metrics import hamming_loss, jaccard_score, f1_score, accuracy_score

X_ml, y_ml = make_multilabel_classification(n_samples=1000, n_features=20, n_classes=5,
                                              n_labels=2, random_state=SEED)
X_ml_tr, X_ml_te, y_ml_tr, y_ml_te = train_test_split(X_ml, y_ml, test_size=0.3, random_state=SEED)

from sklearn.multioutput import MultiOutputClassifier
ml_clf = MultiOutputClassifier(LogisticRegression(max_iter=1000)).fit(X_ml_tr, y_ml_tr)
y_ml_pred = ml_clf.predict(X_ml_te)

print(f"Subset accuracy (exact match) : {accuracy_score(y_ml_te, y_ml_pred):.4f}")
print(f"Hamming loss                  : {hamming_loss(y_ml_te, y_ml_pred):.4f}")
print(f"Jaccard (samples)             : {jaccard_score(y_ml_te, y_ml_pred, average='samples'):.4f}")
print(f"F1 (samples)                  : {f1_score(y_ml_te, y_ml_pred, average='samples'):.4f}")
print(f"F1 (micro)                    : {f1_score(y_ml_te, y_ml_pred, average='micro'):.4f}")
print(f"F1 (macro)                    : {f1_score(y_ml_te, y_ml_pred, average='macro'):.4f}")

### Label ranking metrics (multi-label ordinal)

Si on a des scores pour chaque label, on peut evaluer le **ranking** :


In [ ]:
from sklearn.metrics import label_ranking_average_precision_score, coverage_error, label_ranking_loss

# Probas pour chaque label
y_ml_score = np.array([est.predict_proba(X_ml_te)[:, 1] for est in ml_clf.estimators_]).T

print(f"Label ranking avg precision (LRAP) : {label_ranking_average_precision_score(y_ml_te, y_ml_score):.4f}")
print(f"Coverage error                     : {coverage_error(y_ml_te, y_ml_score):.4f}")
print(f"Label ranking loss                 : {label_ranking_loss(y_ml_te, y_ml_score):.4f}")

---

## 5. Regression

Les metriques de regression mesurent l'**ecart numerique** entre `y_true` et `y_pred`.

| Metrique | Formule | Plage | Sensible aux outliers ? | Echelle |
|---|---|---|---|---|
| **MSE** (Mean Squared Error) | `Σ(y-ŷ)²/N` | [0, ∞[ | ✅ Tres | y² |
| **RMSE** (Root MSE) | `√MSE` | [0, ∞[ | ✅ | y |
| **MAE** (Mean Absolute Error) | `Σ|y-ŷ|/N` | [0, ∞[ | Robuste | y |
| **MedAE** (Median AE) | `median|y-ŷ|` | [0, ∞[ | Tres robuste | y |
| **MAPE** (Mean Absolute % Error) | `Σ|y-ŷ|/|y|/N` | [0, ∞[ | ✅ | % |
| **sMAPE** (Symmetric MAPE) | `Σ|y-ŷ|/((|y|+|ŷ|)/2)/N` | [0, 2] | ✅ | % |
| **MSLE** (Mean Squared Log Error) | `Σ(log(1+y)-log(1+ŷ))²/N` | [0, ∞[ | Modere | log y |
| **R²** (Coefficient determination) | `1 - SS_res/SS_tot` | ]-∞, 1] | ✅ | unitless |
| **R² adjusted** | `1-(1-R²)(n-1)/(n-p-1)` | ]-∞, 1] | ✅ | unitless |
| **Explained variance** | `1 - Var(y-ŷ)/Var(y)` | ]-∞, 1] | ✅ | unitless |
| **Max error** | `max|y-ŷ|` | [0, ∞[ | ✅ ✅ | y |
| **Huber** | MSE pour petites erreurs, MAE pour grandes | [0, ∞[ | Robuste | mixed |
| **Quantile loss (pinball)** | `Σ max(τ(y-ŷ), (τ-1)(y-ŷ))/N` | [0, ∞[ | Asymetrique | y |
| **Tweedie deviance** | family-specific | [0, ∞[ | Depend | depend |

> 🎯 **Regles** :
> - **RMSE** par defaut pour reporter (memes unites que y, penalise les gros ecarts)
> - **MAE** si distribution lourde / outliers presents
> - **MAPE/sMAPE** si scale-invariant requis (y de magnitudes differentes)
> - **R²** pour comparer vs baseline naive (mean predictor → R²=0)
> - **Quantile** pour predire des intervalles


In [ ]:
from sklearn.metrics import (
    mean_squared_error, root_mean_squared_error, mean_absolute_error, median_absolute_error,
    mean_absolute_percentage_error, mean_squared_log_error,
    r2_score, explained_variance_score, max_error,
    mean_pinball_loss, mean_tweedie_deviance,
)

reg_metrics = {
    "MSE":  mean_squared_error(y_reg_te, y_reg_pred),
    "RMSE": root_mean_squared_error(y_reg_te, y_reg_pred),
    "MAE":  mean_absolute_error(y_reg_te, y_reg_pred),
    "MedAE": median_absolute_error(y_reg_te, y_reg_pred),
    # MAPE : eviter la division par 0 en clippant y_true
    "MAPE": mean_absolute_percentage_error(
        np.where(np.abs(y_reg_te) < 1e-6, 1e-6, y_reg_te),
        y_reg_pred,
    ),
    "Max error": max_error(y_reg_te, y_reg_pred),
    "R²":   r2_score(y_reg_te, y_reg_pred),
    "Explained variance": explained_variance_score(y_reg_te, y_reg_pred),
}
pd.DataFrame(reg_metrics, index=["score"]).T

### 5.1 R² ajuste (vs nombre de features)

Le R² normal augmente toujours quand on ajoute une feature. Le **R² ajuste** penalise l'ajout de features non informatives.

`R²_adj = 1 - (1 - R²) × (n - 1) / (n - p - 1)` ou `n` = nb samples, `p` = nb features.


In [ ]:
def r2_adjusted(r2, n, p):
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

n = len(y_reg_te); p = X_reg_te.shape[1]
r2 = r2_score(y_reg_te, y_reg_pred)
print(f"R²        : {r2:.4f}")
print(f"R² adj    : {r2_adjusted(r2, n, p):.4f}  (n={n}, p={p})")

### 5.2 sMAPE (Symmetric MAPE) — utile en forecasting

MAPE classique explose quand `y` proche de 0. sMAPE divise par la **moyenne** de `|y|` et `|ŷ|` → borne [0, 2].


In [ ]:
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / np.where(denom == 0, 1, denom)
    return np.mean(diff)

print(f"sMAPE : {smape(y_reg_te, y_reg_pred):.4f}  (range [0, 2])")

### 5.3 Quantile loss (pinball) — pour prediction intervals

Permet d'optimiser une **prediction de quantile** (p10, p50=median, p90). Utilise dans LightGBM/CatBoost avec `objective='quantile'`.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

# Modeles pour quantiles 10%, 50%, 90%
models_q = {}
for alpha in [0.1, 0.5, 0.9]:
    m = GradientBoostingRegressor(loss="quantile", alpha=alpha, n_estimators=100, random_state=SEED)
    m.fit(X_reg_tr, y_reg_tr)
    models_q[alpha] = m

# Pinball loss par quantile
for alpha, m in models_q.items():
    pred = m.predict(X_reg_te)
    loss = mean_pinball_loss(y_reg_te, pred, alpha=alpha)
    coverage = np.mean(y_reg_te <= pred) if alpha == 0.9 else None
    msg = f"alpha={alpha}, pinball loss = {loss:.4f}"
    if alpha == 0.9:
        msg += f" | coverage(<=q90) = {coverage:.2%}  (cible 90%)"
    print(msg)

### 5.4 Tweedie deviance (insurance, retail)

Loss appropriee pour distributions zero-inflated (claim count, sales). `power` parametre la distribution :
- 0 = Normal (= MSE)
- 1 = Poisson
- 2 = Gamma
- (1, 2) = Compound Poisson-Gamma


In [ ]:
# Y positif requis pour Tweedie
y_pos = np.abs(y_reg_te) + 1
y_pred_pos = np.abs(y_reg_pred) + 1

for power in [0, 1, 1.5, 2]:
    try:
        d = mean_tweedie_deviance(y_pos, y_pred_pos, power=power)
        print(f"power={power}: {d:.4f}")
    except ValueError as e:
        print(f"power={power}: invalid -> {e}")

---

## 6. Ranking et Information Retrieval

Quand l'output est un **classement** de N items (recherche, recommandation), les metriques au-dessus ne suffisent pas. On veut savoir si **les bonnes reponses sont en haut**.

| Metrique | Formule | Plage | Recompense les top |
|---|---|---|---|
| **HitRate@k** | 1 si au moins 1 pertinent dans top-k, 0 sinon | {0, 1} | Faible |
| **Precision@k** | `pertinents dans top-k / k` | [0, 1] | Faible |
| **Recall@k** | `pertinents dans top-k / total pertinents` | [0, 1] | Faible |
| **MRR** (Mean Reciprocal Rank) | `Σ 1/rank_premier_pertinent / N` | [0, 1] | Premier seulement |
| **MAP@k** (Mean Average Precision) | `Σ AP@k(query) / N_queries` | [0, 1] | Moyennement |
| **NDCG@k** (Normalized DCG) | `DCG@k / IDCG@k` | [0, 1] | Tres (log discount) |
| **ERR** (Expected Reciprocal Rank) | Modele cascade | [0, 1] | Tres |
| **Coverage** | % du catalogue jamais recommande | [0, 1] | - |
| **Diversity** | similarite intra-listes (inverse) | [0, 1] | - |

**NDCG** (Normalized Discounted Cumulative Gain) = la metrique reine en search/recsys.

`DCG@k = Σ (2^rel_i - 1) / log2(i + 1)` pour i=1..k
`NDCG@k = DCG@k / IDCG@k` (IDCG = DCG ideal en triant par rel decroissante).


In [ ]:
from sklearn.metrics import ndcg_score, label_ranking_average_precision_score

# Donnees jouet : 5 queries, 10 items, scores et pertinence (0-3)
np.random.seed(SEED)
n_queries, n_items = 5, 10
y_rel = np.random.randint(0, 4, (n_queries, n_items))   # pertinence reelle (0-3)
y_score = np.random.rand(n_queries, n_items)            # scores predits du modele

# NDCG par query et moyen
ndcg_at_5 = ndcg_score(y_rel, y_score, k=5)
ndcg_at_10 = ndcg_score(y_rel, y_score, k=10)
print(f"NDCG@5  : {ndcg_at_5:.4f}")
print(f"NDCG@10 : {ndcg_at_10:.4f}")

### 6.1 Implementation manuelle (MAP@k, MRR, HitRate@k)


In [ ]:
def precision_at_k(relevant, ranked, k):
    return np.mean([item in relevant for item in ranked[:k]])

def recall_at_k(relevant, ranked, k):
    if len(relevant) == 0: return 0.0
    return sum(1 for item in ranked[:k] if item in relevant) / len(relevant)

def hit_rate_at_k(relevant, ranked, k):
    return float(any(item in relevant for item in ranked[:k]))

def average_precision_at_k(relevant, ranked, k):
    if not relevant: return 0.0
    score = 0.0
    n_hits = 0
    for i, item in enumerate(ranked[:k], start=1):
        if item in relevant:
            n_hits += 1
            score += n_hits / i
    return score / min(len(relevant), k)

def reciprocal_rank(relevant, ranked):
    for i, item in enumerate(ranked, start=1):
        if item in relevant: return 1.0 / i
    return 0.0

# Demo
relevant = {2, 5, 7}                   # items pertinents (ground truth)
ranked = [3, 5, 1, 2, 9, 7, 0, 4, 8, 6]  # ordre predit par le modele

for k in [3, 5, 10]:
    print(f"k={k:2d}  P@k={precision_at_k(relevant, ranked, k):.3f}"
          f"  R@k={recall_at_k(relevant, ranked, k):.3f}"
          f"  HitRate@k={hit_rate_at_k(relevant, ranked, k):.0f}"
          f"  MAP@k={average_precision_at_k(relevant, ranked, k):.3f}")
print(f"MRR : {reciprocal_rank(relevant, ranked):.3f}")

---

## 7. Calibration probabiliste

Un modele est **calibre** si quand il dit `p=0.7`, l'evenement arrive 70% du temps. Beaucoup de modeles (RF, GBM, SVM, NN sans modif) sont **mal calibres** par defaut.

**Diagnostic** : reliability diagram + Brier score + ECE.

| Metrique | Formule | Interpretation |
|---|---|---|
| **Brier score** | `Σ(p-y)²/N` | Proper scoring rule, < 0.25 pour binaire |
| **Log loss** | `-Σy log p` | Proper scoring rule, penalise les certitudes fausses |
| **ECE** (Expected Calibration Error) | `Σ|acc(bin) - conf(bin)| × n_bin/N` | < 0.05 = bien calibre |
| **MCE** (Max Calibration Error) | `max|acc(bin) - conf(bin)|` | Worst-case |


In [ ]:
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

# Comparer LogReg (souvent bien calibre) et RF (souvent overconfident)
rf = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_bin_tr, y_bin_tr)
rf_proba = rf.predict_proba(X_bin_te)[:, 1]

# Reliability diagram
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
for name, proba in [("LogReg", y_bin_proba), ("RF (overconfident?)", rf_proba)]:
    frac_pos, mean_pred = calibration_curve(y_bin_te, proba, n_bins=10, strategy="quantile")
    bs = brier_score_loss(y_bin_te, proba)
    ax.plot(mean_pred, frac_pos, "o-", label=f"{name} (Brier={bs:.3f})")
ax.set(xlabel="Mean predicted probability", ylabel="Fraction of positives", title="Reliability diagram")
ax.legend()
plt.show()

### 7.1 Expected Calibration Error (ECE)


In [ ]:
def expected_calibration_error(y_true, y_proba, n_bins=10):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_proba >= bin_edges[i]) & (y_proba < bin_edges[i + 1])
        if mask.sum() == 0: continue
        acc = y_true[mask].mean()
        conf = y_proba[mask].mean()
        ece += (mask.sum() / len(y_true)) * abs(acc - conf)
    return ece

ece_lr = expected_calibration_error(y_bin_te, y_bin_proba)
ece_rf = expected_calibration_error(y_bin_te, rf_proba)
print(f"ECE LogReg : {ece_lr:.4f}")
print(f"ECE RF     : {ece_rf:.4f}")

### 7.2 Calibration post-hoc (Platt, Isotonic)

Si modele mal calibre → re-calibrer avec `CalibratedClassifierCV` :


In [ ]:
cal_iso = CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=SEED),
                                  method="isotonic", cv=5).fit(X_bin_tr, y_bin_tr)
cal_proba_iso = cal_iso.predict_proba(X_bin_te)[:, 1]

cal_sig = CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=SEED),
                                  method="sigmoid", cv=5).fit(X_bin_tr, y_bin_tr)
cal_proba_sig = cal_sig.predict_proba(X_bin_te)[:, 1]

print(f"Brier RF raw       : {brier_score_loss(y_bin_te, rf_proba):.4f}")
print(f"Brier RF isotonic  : {brier_score_loss(y_bin_te, cal_proba_iso):.4f}")
print(f"Brier RF sigmoid   : {brier_score_loss(y_bin_te, cal_proba_sig):.4f}")

---

## 8. Clustering

2 cas :
- **Avec ground truth** : metriques de comparaison labels predits vs vrais
- **Sans ground truth** : metriques internes (cohesion intra-cluster, separation inter-cluster)

| Metrique | Type | Plage | Quand |
|---|---|---|---|
| **Silhouette** | Interne | [-1, 1] | Sans GT, > 0.5 = clusters nets |
| **Davies-Bouldin** | Interne | [0, ∞[ | Sans GT, **plus bas = mieux** |
| **Calinski-Harabasz** | Interne | [0, ∞[ | Sans GT, plus haut = mieux |
| **ARI** (Adjusted Rand Index) | Externe | [-0.5, 1] | Avec GT, ajuste pour le hasard |
| **AMI** (Adjusted Mutual Info) | Externe | [0, 1] | Avec GT |
| **NMI** (Normalized MI) | Externe | [0, 1] | Avec GT |
| **V-measure** | Externe | [0, 1] | Harmonique de homogeneity + completeness |
| **Homogeneity** | Externe | [0, 1] | Chaque cluster contient 1 seule classe ? |
| **Completeness** | Externe | [0, 1] | Chaque classe est dans 1 seul cluster ? |
| **Fowlkes-Mallows** | Externe | [0, 1] | Geometrique de precision+recall |


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, adjusted_mutual_info_score, normalized_mutual_info_score,
    v_measure_score, homogeneity_score, completeness_score, fowlkes_mallows_score,
)

X_clu, y_clu_true = make_blobs(n_samples=1000, n_features=5, centers=4, cluster_std=1.5, random_state=SEED)
kmeans = KMeans(n_clusters=4, random_state=SEED, n_init=10).fit(X_clu)
y_clu_pred = kmeans.labels_

clu_metrics = {
    # Internes (sans GT)
    "Silhouette":        silhouette_score(X_clu, y_clu_pred),
    "Davies-Bouldin":    davies_bouldin_score(X_clu, y_clu_pred),
    "Calinski-Harabasz": calinski_harabasz_score(X_clu, y_clu_pred),
    # Externes (avec GT)
    "ARI":               adjusted_rand_score(y_clu_true, y_clu_pred),
    "AMI":               adjusted_mutual_info_score(y_clu_true, y_clu_pred),
    "NMI":               normalized_mutual_info_score(y_clu_true, y_clu_pred),
    "V-measure":         v_measure_score(y_clu_true, y_clu_pred),
    "Homogeneity":       homogeneity_score(y_clu_true, y_clu_pred),
    "Completeness":      completeness_score(y_clu_true, y_clu_pred),
    "Fowlkes-Mallows":   fowlkes_mallows_score(y_clu_true, y_clu_pred),
}
pd.DataFrame(clu_metrics, index=["score"]).T

### 8.1 Elbow method (choix du k)

Tracer **inertia** vs k. "Coude" = k optimal.


In [ ]:
inertias = []
ks = range(2, 11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X_clu)
    inertias.append(km.inertia_)

plt.plot(ks, inertias, "o-")
plt.xlabel("k"); plt.ylabel("Inertia (within-cluster SSD)")
plt.title("Elbow method")
plt.show()

---

## 9. Survival analysis

Metriques specifiques : on doit gerer la **censure**.

| Metrique | Sens |
|---|---|
| **C-index** (Concordance) | Proba que 2 individus comparables soient bien ordonnes en risque |
| **Brier score (time-dependent)** | Erreur quadratique de S(t) a un temps `t` donne |
| **IBS** (Integrated Brier Score) | Brier integre sur [0, T] |
| **Time-dependent AUC** | AUC `(t)` |
| **Log-rank test** | Difference entre courbes de survie de 2 groupes (p-value) |

> C-index ≥ 0.7 = utile ; ≥ 0.8 = bon ; 0.5 = random.


In [ ]:
# pip install -q scikit-survival lifelines
try:
    from sksurv.metrics import concordance_index_censored, integrated_brier_score
    from sksurv.datasets import load_veterans_lung_cancer
    from sksurv.ensemble import RandomSurvivalForest
    import sklearn

    X_surv, y_surv = load_veterans_lung_cancer()
    # y_surv est un structured array avec ('Status', 'Survival_in_days')
    X_surv = X_surv.select_dtypes(include=[np.number])  # simplifier pour le demo

    rsf = RandomSurvivalForest(n_estimators=50, min_samples_split=10,
                               min_samples_leaf=15, random_state=SEED).fit(X_surv, y_surv)

    pred = rsf.predict(X_surv)  # risk score
    c_index, _, _, _, _ = concordance_index_censored(y_surv["Status"], y_surv["Survival_in_days"], pred)
    print(f"C-index (train, optimistic) : {c_index:.4f}")
except ImportError:
    print("scikit-survival pas installe. Installer avec : uv add --group ds_advanced scikit-survival")

---

## 10. Time Series forecasting

Metriques specifiques aux series temporelles. Standards en competitions (M3, M4, M5).

| Metrique | Formule | Plage | Quand |
|---|---|---|---|
| **MAE / RMSE** | Comme reg | [0, ∞[ | Eval simple |
| **MAPE** | `Σ|y-ŷ|/|y|/N` | [0, ∞[ | Si y > 0 toujours |
| **sMAPE** | symetrique | [0, 200%] | Compet M3/M4 |
| **MASE** (Mean Absolute Scaled Error) | `MAE / MAE_naive` | [0, ∞[ | Scale-invariant, **recommande** |
| **RMSSE** (Root Mean Squared Scaled Error) | `√(MSE / MSE_naive)` | [0, ∞[ | Compet M5 |
| **WAPE** (Weighted APE) | `Σ|y-ŷ| / Σ|y|` | [0, ∞[ | Aggregation hierarchique |
| **OWA** (Overall Weighted Average) | Mean sMAPE + MASE normalise | [0, ∞[ | M4 |
| **Pinball loss** | `Σ max(τe, (τ-1)e)/N` | [0, ∞[ | Forecast quantile |
| **Coverage** | `P(y in [q_low, q_high])` | [0, 1] | Validation intervalle |
| **Winkler score** | Interval score | [0, ∞[ | Forecast interval combine |


In [ ]:
# Demo : MASE, sMAPE
def mase(y_true, y_pred, y_train, seasonality=1):
    """Mean Absolute Scaled Error (Hyndman 2006).
    Compare MAE du modele a MAE du baseline 'naive seasonal' sur le TRAIN."""
    n = len(y_train)
    mae_naive = np.mean(np.abs(np.array(y_train[seasonality:]) - np.array(y_train[:-seasonality])))
    mae_model = np.mean(np.abs(np.array(y_true) - np.array(y_pred)))
    return mae_model / mae_naive

def rmsse(y_true, y_pred, y_train, seasonality=1):
    n = len(y_train)
    mse_naive = np.mean((np.array(y_train[seasonality:]) - np.array(y_train[:-seasonality]))**2)
    mse_model = np.mean((np.array(y_true) - np.array(y_pred))**2)
    return np.sqrt(mse_model / mse_naive)

def wape(y_true, y_pred):
    return np.sum(np.abs(np.array(y_true) - np.array(y_pred))) / np.sum(np.abs(y_true))

# Series jouet : tendance + saisonnalite + bruit
np.random.seed(SEED)
n_train, n_test = 200, 50
t = np.arange(n_train + n_test)
y = 100 + 0.5 * t + 20 * np.sin(2 * np.pi * t / 12) + np.random.randn(len(t)) * 5
y_train_ts, y_test_ts = y[:n_train], y[n_train:]

# Predictions naives
y_naive = np.full(n_test, y_train_ts[-1])               # repeter derniere valeur
y_seasonal_naive = y_train_ts[-12:][:n_test % 12] if n_test < 12 else np.tile(y_train_ts[-12:], (n_test // 12) + 1)[:n_test]

print(f"sMAPE  naive          : {smape(y_test_ts, y_naive):.4f}")
print(f"sMAPE  seasonal naive : {smape(y_test_ts, y_seasonal_naive):.4f}")
print(f"MASE   naive          : {mase(y_test_ts, y_naive, y_train_ts, seasonality=1):.4f}")
print(f"MASE   seasonal naive : {mase(y_test_ts, y_seasonal_naive, y_train_ts, seasonality=12):.4f}")
print(f"RMSSE  seasonal naive : {rmsse(y_test_ts, y_seasonal_naive, y_train_ts, seasonality=12):.4f}")
print(f"WAPE   seasonal naive : {wape(y_test_ts, y_seasonal_naive):.4f}")

### 10.1 Interval forecast — Winkler score

Pour evaluer un **intervalle de prediction** `[L, U]` au niveau de confiance `1-α`.

`W_α(L, U, y) = (U - L) + (2/α)·(L-y) · 1{y<L} + (2/α)·(y-U) · 1{y>U}`

Penalise les intervalles larges ET les non-couvertures.


In [ ]:
def winkler_score(y, lower, upper, alpha=0.1):
    width = upper - lower
    penalty_low = (2 / alpha) * np.maximum(lower - y, 0)
    penalty_up = (2 / alpha) * np.maximum(y - upper, 0)
    return np.mean(width + penalty_low + penalty_up)

# Demo
lower = y_test_ts - 10
upper = y_test_ts + 10
coverage = np.mean((y_test_ts >= lower) & (y_test_ts <= upper))
ws = winkler_score(y_test_ts, lower, upper, alpha=0.1)
print(f"Coverage : {coverage:.2%}  (cible : 90%)")
print(f"Winkler  : {ws:.4f}  (plus bas = mieux)")

---

## 11. NLP — metriques de generation

| Metrique | Type | Bon |
|---|---|---|
| **BLEU** | N-gram overlap | Traduction, surface |
| **ROUGE-N/L/W** | Recall n-gram | Summarization |
| **METEOR** | Synonymes + stems | Traduction |
| **chrF** | Char n-grams | Multilingue |
| **BERTScore** | Embeddings BERT | Semantique |
| **BLEURT** | BERT fine-tune sur ratings | Semantique + qualite |
| **COMET** | Cross-lingual eval | Traduction |
| **MAUVE** | Distribution gap LM | Generation libre |
| **Perplexity** | Proba conditionnelle | Eval LM intrinseque |

> **Note** : BLEU et ROUGE sont **lexicaux** (matching exact). BERTScore est **semantique** (paraphrase OK). Pour LLMs modernes, BERTScore / BLEURT recommandes.


In [ ]:
# pip install -q sacrebleu rouge-score bert-score nltk
try:
    import sacrebleu

    refs = [["The cat sits on the mat."]]
    hyps = ["The cat is on the mat."]
    bleu = sacrebleu.corpus_bleu(hyps, refs)
    print(f"BLEU  : {bleu.score:.2f}  (echelle 0-100)")

    # chrF
    chrf = sacrebleu.corpus_chrf(hyps, refs)
    print(f"chrF  : {chrf.score:.2f}")

    # TER (Translation Edit Rate)
    ter = sacrebleu.corpus_ter(hyps, refs)
    print(f"TER   : {ter.score:.2f}  (plus bas = mieux)")
except ImportError:
    print("Installer : pip install -q sacrebleu")

### 11.1 ROUGE (summarization)


In [ ]:
try:
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    ref = "The cat sat on the mat in the warm afternoon sun."
    hyp = "The cat was sitting on the mat in the sun."
    scores = scorer.score(ref, hyp)
    for k, v in scores.items():
        print(f"{k} : P={v.precision:.3f}  R={v.recall:.3f}  F1={v.fmeasure:.3f}")
except ImportError:
    print("Installer : pip install -q rouge-score")

### 11.2 BERTScore (semantique)

Tres bon pour LLM eval — encode ref et hyp avec un BERT et compare les embeddings token a token.


In [ ]:
# Demo (pseudo-code car ~ 500 MB de modele a DL)
# from bert_score import score
# refs = ["The cat sits on the mat."]
# hyps = ["A feline rests on the rug."]
# P, R, F1 = score(hyps, refs, lang="en", model_type="bert-base-uncased")
# print(f"BERTScore F1 : {F1.mean().item():.4f}")
print("BERTScore : decommenter pour usage (telecharge ~500 MB de modele)")

### 11.3 Perplexity (LM intrinseque)

PPL d'un texte sous un LM. **Plus bas = mieux**. Utilise pour comparer modeles de langue.

`PPL(x) = exp(-1/N · Σ log P(x_i | x_<i))`


In [ ]:
def perplexity(log_probs):
    """log_probs : log P(x_i | x_<i) pour chaque token. Renvoie PPL."""
    return np.exp(-np.mean(log_probs))

# Demo synthetique : un modele "bon" et "mauvais"
np.random.seed(SEED)
log_probs_good = np.random.normal(loc=-2.0, scale=0.5, size=100)
log_probs_bad = np.random.normal(loc=-5.0, scale=1.0, size=100)
print(f"PPL bon modele     : {perplexity(log_probs_good):.2f}")
print(f"PPL mauvais modele : {perplexity(log_probs_bad):.2f}")

---

## 12. Computer Vision

Selon la tache :

### 12.1 Detection (object detection)

| Metrique | Formule | Standard |
|---|---|---|
| **IoU** (Intersection over Union) | `\|A∩B\| / \|A∪B\|` | Base de tout |
| **mAP@0.5** | mean AP avec IoU threshold 0.5 | PASCAL VOC |
| **mAP@[.5:.95]** | mAP moyenne sur IoU 0.5-0.95 par pas de 0.05 | COCO standard |
| **AR** (Average Recall) | Recall moyen par # detections | COCO |


In [ ]:
def iou_bbox(box1, box2):
    """Boxes en format [x1, y1, x2, y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    return inter / (area1 + area2 - inter)

box_gt = [10, 10, 50, 50]
box_pred = [15, 15, 55, 60]
print(f"IoU : {iou_bbox(box_gt, box_pred):.4f}")

### 12.2 Segmentation

| Metrique | Sens |
|---|---|
| **Pixel accuracy** | `correct_pixels / total_pixels` |
| **Mean IoU (mIoU)** | IoU moyen par classe |
| **Dice / F1 coeff** | `2|A∩B| / (|A|+|B|)` = 2·IoU/(1+IoU) |
| **Boundary F1** | F1 sur les contours seulement |


In [ ]:
def dice_coefficient(y_true, y_pred):
    """Pour masks binaires (numpy)."""
    intersection = (y_true & y_pred).sum()
    return 2 * intersection / (y_true.sum() + y_pred.sum() + 1e-10)

def iou_mask(y_true, y_pred):
    inter = (y_true & y_pred).sum()
    union = (y_true | y_pred).sum()
    return inter / (union + 1e-10)

# Demo : masque 10x10
y_true = np.zeros((10, 10), dtype=bool); y_true[2:7, 2:7] = True
y_pred = np.zeros((10, 10), dtype=bool); y_pred[3:8, 3:8] = True

print(f"Dice : {dice_coefficient(y_true, y_pred):.4f}")
print(f"IoU  : {iou_mask(y_true, y_pred):.4f}")

### 12.3 Image quality

| Metrique | Sens | Plage |
|---|---|---|
| **PSNR** (Peak Signal-to-Noise Ratio) | `10·log10(MAX²/MSE)` | dB, > 30 = bon |
| **SSIM** (Structural Similarity) | luminance × contrast × structure | [-1, 1], 1 = identique |
| **LPIPS** (Learned Perceptual Image Patch Sim.) | Distance d'embeddings VGG | [0, ∞[, plus bas = mieux |
| **FID** (Frechet Inception Distance) | Distance feat distributions (Inception) | [0, ∞[, plus bas = mieux |


In [ ]:
def psnr(img1, img2, max_val=255):
    mse = np.mean((img1.astype(float) - img2.astype(float))**2)
    if mse == 0: return float("inf")
    return 10 * np.log10(max_val**2 / mse)

# Demo
np.random.seed(SEED)
img_ref = np.random.randint(0, 256, (64, 64)).astype(np.uint8)
img_noisy = img_ref + np.random.randint(-20, 20, img_ref.shape)
img_noisy = np.clip(img_noisy, 0, 255).astype(np.uint8)

print(f"PSNR : {psnr(img_ref, img_noisy):.2f} dB")

# SSIM via scikit-image
try:
    from skimage.metrics import structural_similarity as ssim
    print(f"SSIM : {ssim(img_ref, img_noisy):.4f}")
except ImportError:
    print("Installer scikit-image : pip install -q scikit-image")

---

## 13. LLM / RAG — Generation, retrieval, faithfulness

Pour les LLMs et systemes RAG, les metriques classiques NLP (BLEU) sont **insuffisantes**. On veut savoir si la reponse est **factuelle**, **pertinente**, **fondee dans le contexte**.

### Metriques `ragas` (RAG evaluation)

| Metrique | Sens | Nécessite |
|---|---|---|
| **Faithfulness** | La reponse est-elle ancree dans le contexte ? | LLM-as-judge |
| **Answer relevancy** | La reponse repond-elle a la question ? | LLM-as-judge |
| **Context precision** | Les chunks recuperes sont-ils pertinents ? | Ranking |
| **Context recall** | Tous les chunks pertinents ont-ils ete recuperes ? | GT |
| **Answer correctness** | Reponse vs ground-truth | GT + LLM-as-judge |

### LLM-as-judge

Au lieu d'une metrique calculable, on demande a un LLM puissant (GPT-4, Claude Opus) de noter la reponse selon une rubrique. Methodes :
- **G-Eval** (Liu et al. 2023) — CoT + ratings
- **MT-Bench** (Zheng et al. 2023) — 80 questions multi-turn
- **Prometheus** — Open-source juge fine-tune

### Exemple (pseudo-code ragas)


In [ ]:
# pip install -q ragas datasets

# from datasets import Dataset
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

# data = {
#     "question":     ["Quelle est la capitale de la France ?"],
#     "answer":       ["La capitale de la France est Paris."],
#     "contexts":     [["Paris est la capitale et plus grande ville de la France."]],
#     "ground_truth": ["Paris"],
# }
# ds = Dataset.from_dict(data)
# result = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
# print(result)

# Sortie type :
# {'faithfulness': 0.95, 'answer_relevancy': 0.92, 'context_precision': 1.0, 'context_recall': 1.0}
print("ragas : necessite OPENAI_API_KEY ou autre LLM. Voir NLP_LangChain_RAG.ipynb")

### 13.1 G-Eval (LLM as judge custom)


In [ ]:
# Template de prompt G-Eval
GEVAL_TEMPLATE = '''
Tu es un evaluateur expert. Note la reponse selon le critere ci-dessous.

Question : {question}
Reponse a evaluer : {answer}
Contexte de reference : {context}

CRITERE : {criterion}
DESCRIPTION DU CRITERE : {criterion_desc}

ETAPES :
1. Lis attentivement la reponse
2. Compare au contexte
3. Evalue selon le critere sur 1-5
4. Donne la note seule (entier 1-5)

Note (1-5) :
'''
print("Template de prompt pour LLM-as-judge defini.")
print("Usage : envoyer ce prompt a GPT-4 / Claude Opus / un LLM local puissant")

---

## 14. Fairness

Quand un modele a un impact business / social, on doit verifier qu'il **traite equitablement** les sous-groupes (genre, age, ethnie...).

| Metrique | Definition |
|---|---|
| **Disparate impact (DI)** | `P(ŷ=1\|A=unprivileged) / P(ŷ=1\|A=privileged)`. Cible 1.0, alerte si < 0.8 (regle des 4/5) |
| **Demographic parity** | `P(ŷ=1\|A=0) ≈ P(ŷ=1\|A=1)` |
| **Equal opportunity** | `TPR(A=0) ≈ TPR(A=1)` |
| **Equalized odds** | TPR et FPR egaux entre groupes |
| **Predictive parity** | Precision egale entre groupes |
| **Calibration parity** | Calibration egale entre groupes |

> ⚠️ **Impossibility theorem** (Chouldechova 2017) : sauf cas degenere, **on ne peut pas avoir tout en meme temps**. Il faut choisir laquelle de ces fairness compte le plus dans le contexte business/legal.


In [ ]:
# Exemple manuel sur dataset jouet : groupe sensible 'sex' encode dans X
# Pour la demo on utilise une feature aleatoire comme groupe
A = np.random.choice([0, 1], size=len(y_bin_te))  # 0 = unprivileged, 1 = privileged

def disparate_impact(y_pred, A):
    p_1 = np.mean(y_pred[A == 1])
    p_0 = np.mean(y_pred[A == 0])
    if p_1 == 0: return float("inf")
    return p_0 / p_1

def demographic_parity_diff(y_pred, A):
    return np.mean(y_pred[A == 1]) - np.mean(y_pred[A == 0])

def equal_opportunity_diff(y_true, y_pred, A):
    """Difference de TPR entre groupes."""
    tpr_1 = np.mean(y_pred[(A == 1) & (y_true == 1)]) if ((A == 1) & (y_true == 1)).any() else 0
    tpr_0 = np.mean(y_pred[(A == 0) & (y_true == 1)]) if ((A == 0) & (y_true == 1)).any() else 0
    return tpr_1 - tpr_0

print(f"Disparate impact      : {disparate_impact(y_bin_pred, A):.4f}  (cible 1.0, alerte si < 0.8)")
print(f"Demographic parity Δ  : {demographic_parity_diff(y_bin_pred, A):.4f}  (cible 0)")
print(f"Equal opportunity Δ   : {equal_opportunity_diff(y_bin_te, y_bin_pred, A):.4f}  (cible 0)")

### 14.1 Outils dedies : `fairlearn`, `aif360`

```python
# pip install -q fairlearn
# from fairlearn.metrics import (
#     demographic_parity_difference, demographic_parity_ratio,
#     equalized_odds_difference, MetricFrame, selection_rate,
# )
# from fairlearn.metrics import false_positive_rate, true_positive_rate

# # Rapport multi-metriques par groupe
# mf = MetricFrame(
#     metrics={
#         "accuracy":  accuracy_score,
#         "selection_rate": selection_rate,
#         "FPR": false_positive_rate,
#         "TPR": true_positive_rate,
#     },
#     y_true=y_bin_te, y_pred=y_bin_pred,
#     sensitive_features=A,
# )
# print(mf.by_group)
```


---

## 15. Tableau de selection — quelle metrique pour quel probleme ?

| Probleme | Metrique primaire | Metriques secondaires |
|---|---|---|
| **Classif binaire equilibree** | Accuracy ou F1 | ROC-AUC |
| **Classif binaire desequilibree** | F1 ou PR-AUC | MCC, Recall@P=X |
| **Classif binaire ordinale** | Quadratic kappa | F1 macro |
| **Detection rare event** (1:1000+) | PR-AUC | Precision@k |
| **Classif multi-classe** | F1 macro / weighted | Confusion matrix, top-k accuracy |
| **Multi-label** | F1 (samples) | Hamming loss, LRAP |
| **Reg avec outliers** | MAE / MedAE | RMSE pour reporting |
| **Reg sans outliers** | RMSE / R² | MAPE si scale-invariant |
| **Reg de comptages (zero-inflated)** | Tweedie / Poisson deviance | Pinball loss |
| **Forecast TS** | MASE / RMSSE | sMAPE, WAPE |
| **Recommandation top-k** | NDCG@k | MAP@k, MRR |
| **IR / Search** | NDCG@10 | MRR, MAP, P@10 |
| **Clustering avec GT** | ARI / AMI | V-measure |
| **Clustering sans GT** | Silhouette | Davies-Bouldin |
| **Survival** | C-index | IBS |
| **Detection anomalie** | PR-AUC, P@k | ROC-AUC |
| **Calibration** | Brier score, ECE | Reliability diagram |
| **Generation NLP** | BERTScore, BLEURT | BLEU, ROUGE pour bench |
| **RAG / QA** | Faithfulness, Answer Relevancy (ragas) | LLM-as-judge |
| **Vision detection** | mAP@[.5:.95] | mAP@0.5, AR |
| **Vision segmentation** | mIoU | Dice, Boundary F1 |
| **Image quality** | LPIPS / SSIM | PSNR |
| **Fairness** | Equal opportunity diff | Disparate impact, equalized odds |


---

## 16. Pieges generaux et bonnes pratiques

### Anti-patterns courants

| ❌ Anti-pattern | ✅ Correctif |
|---|---|
| Accuracy sur classes desequilibrees | F1, MCC, balanced accuracy, PR-AUC |
| ROC-AUC sur desequilibre extreme (1:10000) | PR-AUC |
| MAPE quand y peut etre 0 | sMAPE ou MASE |
| Optimiser MSE puis reporter MAE | Optimiser et reporter la meme metrique |
| Comparer modeles avec metriques differentes | Standardiser une metrique business |
| Une seule metrique pour decider | Toujours plusieurs angles (precision + recall + calibration) |
| F1 macro avec une classe a 5 samples | Stratifier le CV et reporter par classe |
| Tester sur train (data leakage) | Toujours hold-out / CV |
| Pas d'intervalle de confiance | Bootstrap ou CV folds variance |
| Confondre R² (regression) et accuracy | Sont differents (R² peut etre negatif) |
| BLEU pour generation creative | BERTScore / LLM-as-judge |
| Comparer mAP@0.5 et mAP@[.5:.95] | Toujours specifier le threshold |
| ECE comme seule mesure de calibration | Compleeter avec Brier + reliability diagram |
| Optimiser fairness sans business impact | Considerer impossibility theorem |

### Statistical significance

Quand on compare 2 modeles, **toujours rapporter un intervalle de confiance** :


In [ ]:
from scipy import stats

# Bootstrap de la metrique
def bootstrap_metric(y_true, y_pred, metric_fn, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        scores.append(metric_fn(y_true[idx], y_pred[idx]))
    return np.percentile(scores, [2.5, 50, 97.5])

low, mid, high = bootstrap_metric(y_bin_te, y_bin_pred, accuracy_score)
print(f"Accuracy : {mid:.4f}  [95% CI : {low:.4f}, {high:.4f}]")

# Test de significativite entre 2 modeles (McNemar pour classification)
from statsmodels.stats.contingency_tables import mcnemar

# Sup à test pour vraie comparaison de modeles
clf2 = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_bin_tr, y_bin_tr)
y_pred2 = clf2.predict(X_bin_te)

# Table 2x2 : (modele1 correct ?, modele2 correct ?)
correct1 = (y_bin_pred == y_bin_te)
correct2 = (y_pred2 == y_bin_te)
table = pd.crosstab(correct1, correct2).values
result = mcnemar(table, exact=False, correction=True)
print(f"\nMcNemar test (modele 1 vs 2) : statistic={result.statistic:.4f}, p-value={result.pvalue:.4f}")

### Bootstrap pour difference de metriques entre modeles

Plus general que McNemar : on bootstrap la **difference** de metriques.


In [ ]:
def bootstrap_diff(y_true, pred1, pred2, metric_fn, n_bootstrap=1000, seed=42):
    rng = np.random.default_rng(seed)
    diffs = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        diffs.append(metric_fn(y_true[idx], pred1[idx]) - metric_fn(y_true[idx], pred2[idx]))
    low, mid, high = np.percentile(diffs, [2.5, 50, 97.5])
    return mid, (low, high)

mid, (lo, hi) = bootstrap_diff(y_bin_te, y_bin_pred, y_pred2, accuracy_score)
print(f"Δ accuracy (LR - RF) : {mid:+.4f}  [95% CI : {lo:+.4f}, {hi:+.4f}]")
print(f"Significatif ? {'Oui' if (lo > 0 or hi < 0) else 'Non'}")

---

## 17. References

### Documentation officielle
- **scikit-learn metrics** : https://scikit-learn.org/stable/modules/model_evaluation.html
- **scikit-survival metrics** : https://scikit-survival.readthedocs.io/en/stable/api/metrics.html
- **statsmodels diagnostics** : https://www.statsmodels.org/stable/diagnostic.html
- **sacrebleu** (NLP) : https://github.com/mjpost/sacrebleu
- **HuggingFace evaluate** : https://huggingface.co/docs/evaluate
- **ragas** (RAG eval) : https://docs.ragas.io/
- **fairlearn** : https://fairlearn.org/
- **aif360** (IBM fairness) : https://aif360.res.ibm.com/

### Papers fondateurs
- Hyndman & Koehler (2006). *Another look at measures of forecast accuracy* (MASE)
- Chinchor (1992). *MUC-4 evaluation metrics* (precision/recall en NLP)
- Powers (2011). *Evaluation: From Precision, Recall and F-Measure to ROC, Informedness, Markedness and Correlation* (MCC)
- Chouldechova (2017). *Fair prediction with disparate impact* (impossibility)
- Zhang et al. (2019). *BERTScore: Evaluating Text Generation with BERT*
- Sellam et al. (2020). *BLEURT: Learning Robust Metrics for Text Generation*
- Liu et al. (2023). *G-Eval: NLG Evaluation using GPT-4*

### Cheat-sheets externes
- Papers with code — Metrics : https://paperswithcode.com/methods/category/evaluation-metrics
- Kaggle metrics guide : https://www.kaggle.com/code/dansbecker/evaluation-metrics

### Voir aussi (dans cette collection)
- [ML_Explication_Feature_Importance_Selection.ipynb](./ML_Explication_Feature_Importance_Selection.ipynb) — SHAP, LIME, displays sklearn (Calibration, ROC, PR, Confusion)
- [ML_Regression_Classification_CV_GridSearch.ipynb](./ML_Regression_Classification_CV_GridSearch.ipynb) — usage des metriques dans `scoring=` GridSearch
- [DS_Survival_Analysis.ipynb](./DS_Survival_Analysis.ipynb) — C-index et IBS detailles
- [DS_Recommender_Systems.ipynb](./DS_Recommender_Systems.ipynb) — metriques recsys (NDCG, MAP@k)
- [AI_Prompt_Engineering.ipynb](./AI_Prompt_Engineering.ipynb) — eval LLMs
- [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) — metriques de drift detection


<!-- ENRICH_2025_V1 -->

## 🔥 Metriques modernes 2024-2025

### 1. **Conformal Prediction** — incertitude **garantie**

Pour n'importe quel modele black-box, genere des **intervalles de prediction valides** (couvre la vraie valeur avec proba `1-α` garantie).

```python
# pip install mapie
from mapie.regression import MapieRegressor
mapie = MapieRegressor(estimator=model, method="cv_plus", cv=5)
mapie.fit(X_train, y_train)
y_pred, y_pis = mapie.predict(X_test, alpha=0.1)   # 90% intervals
# y_pis shape : (n, 2, 1) — lower / upper bounds
```

**Avantage** : pas de hypothese distributionnelle, marche sur n'importe quel modele.

### 2. **LLM-as-judge** systematique (`G-Eval`, `Prometheus`)

Pour evaluer la qualite de generation LLM sans humain :

```python
GEVAL_PROMPT = '''
Note la reponse sur le criteria '{criterion}' (1-5).
Question : {question}
Reponse : {answer}
Note (1-5) :
'''
# Envoyer a GPT-4 ou Claude Opus pour scoring automatique
# G-Eval (Liu et al. 2023) : prompt CoT + LLM puissant
```

### 3. **`ragas` v0.2+** — metriques RAG modernes

- **Faithfulness** : reponse ancree dans contexte
- **Answer Relevancy** : reponse repond a la question
- **Context Precision/Recall**
- **Answer Correctness** (vs ground truth)
- **Context Entity Recall** (nouveau v0.2)
- **Topic Adherence** (nouveau v0.2)

### 4. **Calibration via `CalibrationDisplay` sklearn**

Reliability diagram natif sklearn :
```python
from sklearn.calibration import CalibrationDisplay
CalibrationDisplay.from_estimator(model, X_test, y_test, n_bins=10)
```

### 5. **`MetricFrame` fairlearn** — eval par groupe

```python
from fairlearn.metrics import MetricFrame
mf = MetricFrame(metrics={"acc": accuracy_score, "f1": f1_score},
                  y_true=y, y_pred=pred, sensitive_features=df["gender"])
print(mf.by_group)
print(mf.difference())   # max - min entre groupes
```

### 6. **`torchmetrics`** — metriques GPU pour DL

`torchmetrics.AUROC`, `torchmetrics.F1Score` etc. — natif PyTorch, multi-GPU compatible.

```python
from torchmetrics import AUROC
metric = AUROC(task="binary").to(device)
metric.update(preds, target)
auc = metric.compute()
```

### 7. **TS metrics moderne — `pinball_loss` (quantile forecasting)**

Pour evaluer un **forecast d'intervalle** (p10, p50, p90), utiliser pinball loss au lieu de RMSE.

```python
from sklearn.metrics import mean_pinball_loss
for alpha in [0.1, 0.5, 0.9]:
    loss = mean_pinball_loss(y_true, y_pred_alpha, alpha=alpha)
```

### 8. **`OWA`** (Overall Weighted Average) — compet M4/M5

Normalise sMAPE + MASE vs Naive2 baseline. Standard moderne pour evaluer un forecaster generaliste.

## 🔥 Tendances 2025

- **Calibration intrinseque** des LLMs (Brier score sur leur confidence)
- **Faithfulness scoring** systematique pour RAG (`ragas`)
- **Conformal prediction** popularise (intervals garantis)
- **Fairness audits** reglementaires (UE AI Act 2025) → MetricFrame, AIF360
- **Counterfactual evaluation** (effets causals plutot que correlations) → econml, dowhy
